# Notebook 3: XGBoost Training Pipeline & Model Registry
**Author:** Guillén Concepción (Senior Data Scientist & MLOps Engineer)

This training pipeline reads historical feature view data from **Hopsworks Feature Store**, performs time-series cross-validation split, trains an **XGBoost Regressor** to predict PM2.5 concentrations, evaluates MAE/RMSE/R² metrics, and registers the trained model artifact in **Hopsworks Model Registry**.

In [ ]:
import os
import sys
import json
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from src.config import TARGET_COL, MODEL_NAME, MODELS_DIR
from src.features import get_feature_names
from src.hopsworks_utils import FeatureStoreManager, ModelRegistryManager

## 1. Extract Feature View from Hopsworks Store

In [ ]:
fs_manager = FeatureStoreManager()
df = fs_manager.read_feature_group()
df.sort_values("timestamp", inplace=True)
print(f"Loaded dataset shape: {df.shape}")
df.head()

## 2. Chronological Time-Series Train/Test Split

In [ ]:
feature_cols = get_feature_names(df)
df_clean = df.dropna(subset=[TARGET_COL] + feature_cols).reset_index(drop=True)

split_idx = int(len(df_clean) * 0.80)
X_train = df_clean.iloc[:split_idx][feature_cols]
y_train = df_clean.iloc[:split_idx][TARGET_COL]
X_test = df_clean.iloc[split_idx:][feature_cols]
y_test = df_clean.iloc[split_idx:][TARGET_COL]

print(f"Train size: {X_train.shape}, Test size: {X_test.shape}")

## 3. Train XGBoost Regressor Model

In [ ]:
print("Training XGBoost Regressor model...")
xgb_model = xgb.XGBRegressor(
    n_estimators=350,
    learning_rate=0.03,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(
    X_train,
    y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)

preds = np.clip(xgb_model.predict(X_test), a_min=0, a_max=None)
mae = mean_absolute_error(y_test, preds)
rmse = np.sqrt(mean_squared_error(y_test, preds))
r2 = r2_score(y_test, preds)

metrics = {"mae": round(mae, 4), "rmse": round(rmse, 4), "r2": round(r2, 4)}
print(f"🎯 XGBoost Model Validation Performance:")
print(f"   • MAE : {mae:.3f} ug/m3")
print(f"   • RMSE: {rmse:.3f}")
print(f"   • R2  : {r2:.3f}")

## 4. Save Artifacts & Register Model in Hopsworks

In [ ]:
mr_manager = ModelRegistryManager(fs_manager)
registered_path = mr_manager.save_model(xgb_model, metrics, model_name=MODEL_NAME)
print(f"✅ Successfully registered XGBoost model to Hopsworks Model Registry at {registered_path}!")